In [ ]:
import pandas as pd
import numpy as np
import gamspy as gs

# 1. LOAD AND PREPROCESS DATA
df = pd.read_excel("dataset_output.xlsx")
df = df.dropna(subset=['id']).copy()

ind_labels = [f"ind_{idx}" for idx in df.index]
transit_labels = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L"] 

beta = {
    'ASC_OPT_OUT': -0.014063, 
    'ASC_A': 0.0,            
    'ASC_B': 0.035173, 
    'ASC_C': -0.049251, 
    'ASC_D': -0.045553, 
    'ASC_E': 0.233853, 
    'ASC_F': -0.305134, 
    'ASC_G': 0.269048,
    'ASC_H': 0.026862,
    'ASC_I': -0.216403,
    'ASC_J': 0.329965,
    'ASC_K': -0.105195,
    'ASC_L': -0.087918,
    'B_DIST': -0.050087,  
    'B_SHELTER': 0.639126, 
    'B_RAINY': -0.379045,      
    'B_DIST_RAINY': -0.017229
}

# 2. PRE-COMPUTE CONSTANT RATIOS FOR OPEN STATE (2D Matrices)
v_optout = beta['ASC_OPT_OUT']
kappa_matrix = np.zeros((len(df), len(transit_labels)))
omega_matrix = np.zeros((len(df), len(transit_labels)))

for j_idx, letter in enumerate(transit_labels):
    dist = df[f"distance_{letter}"].values
    rainy = df["rainy"].values
    has_shelter_attr = df[f"shelter_{letter}"].values 
    asc = beta[f"ASC_{letter}"]
    
    v = (
        asc 
        + beta['B_DIST'] * dist 
        + beta['B_SHELTER'] * has_shelter_attr 
        + beta['B_RAINY'] * rainy 
        + beta['B_DIST_RAINY'] * dist * rainy
    )
    
    # Pre-compute only for the open state scenario
    kappa_matrix[:, j_idx] = np.exp(v) / np.exp(v_optout)
    omega_matrix[:, j_idx] = np.exp(v) / (np.exp(v_optout) + np.exp(v))

# 3. INITIALIZE GAMSPY CONTAINER & SETS
m = gs.Container()

i = gs.Set(m, name="i", records=ind_labels)
j = gs.Set(m, name="j", records=transit_labels)

kappa = gs.Parameter(m, name="kappa", domain=[i, j], records=kappa_matrix)
omega = gs.Parameter(m, name="omega", domain=[i, j], records=omega_matrix)

# 4. VARIABLES & MILP LINEAR FORMULATION (2D Layout)
X = gs.Variable(m, name="X", domain=[i, j], type="Positive")      
X_tilde = gs.Variable(m, name="X_tilde", domain=[i], type="Positive") 
Y = gs.Variable(m, name="Y", domain=[j], type="Binary")         
total_transit_share = gs.Variable(m, name="total_transit_share", type="Free")

eq_prob_conservation = gs.Equation(m, name="eq_prob_conservation", domain=[i])
eq_bound1 = gs.Equation(m, name="eq_bound1", domain=[i, j])
eq_bound2 = gs.Equation(m, name="eq_bound2", domain=[i, j])
eq_bound3 = gs.Equation(m, name="eq_bound3", domain=[i, j])
eq_budget = gs.Equation(m, name="eq_budget")
eq_objective = gs.Equation(m, name="eq_objective")

eq_prob_conservation[i] = X_tilde[i] + gs.Sum(j, X[i, j]) == 1

eq_bound1[i, j] = X[i, j] <= omega[i, j] * Y[j]
eq_bound2[i, j] = X[i, j] <= kappa[i, j] * X_tilde[i]
eq_bound3[i, j] = X_tilde[i] - (1 / kappa[i, j]) * X[i, j] + Y[j] <= 1

eq_budget[...] = gs.Sum(j, Y[j]) <= 8
eq_objective[...] = total_transit_share == gs.Sum((i, j), X[i, j])

# 5. EXECUTE MIP OPTIMIZATION SOLVER
station_mip = gs.Model(
    m,
    name="station_mip",
    equations=[
        eq_prob_conservation, eq_bound1, eq_bound2, eq_bound3, 
        eq_budget, eq_objective
    ],
    problem="MIP",
    sense="MAX",
    objective=total_transit_share
)

station_mip.solve(solver="CPLEX")
print(f"Maximized Total Transit Share: {(station_mip.objective_value / 1000 * 100):.2f}%")
print("\nOptimal Station Opening:")
print(Y.records.set_index("j")["level"])


Maximized Total Transit Share: 74.34%

Optimal Station Opening Matrix:
j
A    1.0
B    1.0
C    1.0
D    1.0
E    1.0
F    0.0
G    1.0
H    1.0
I    0.0
J    1.0
K    0.0
L    0.0
Name: level, dtype: float64
